# Programmatic Magnetic Resonance Fingerprinting Optimization

### Orthogonality 📐

The accuracy and reliability of parameter estimation in MRF fundamentally depend on how distinguishable different signal fingerprints are from one another, leading to the critical aspect of signal orthogonality. When signal evolutions corresponding to different tissue parameters are highly orthogonal (i.e., minimally correlated), the matching process can more reliably distinguish between tissues with similar properties. Lower signal orthogonality for signal fingerprints with different relaxometric origins can lead to increased parameter estimation errors and reduced robustness to noise.


### Cramér-Rao Lower Bound (CRLB) ⛓

The Cramér-Rao Lower Bound (CRLB) provides a theoretical framework for understanding the best possible precision achievable in parameter estimation. In the context of MRF, the CRLB quantifies the minimum variance (uncertainty) in estimating parameters from the signal vectors. The inverse of the Fisher Information Matrix yields the CRLB, with diagonal elements representing the variance lower bounds for each parameter. Importantly, off-diagonal elements reveal correlations between parameter estimates. When parameters are highly correlated, they cannot be estimated independently with high precision.

### Forward Model Extended Phase Graph (EPG) 🌀

To optimize MRF acquisition parameters, we require a forward model that can accurately simulate the complex signal evolution during the sequence. The Extended Phase Graph (EPG, see e.g. [Weigel 2015](https://doi.org/10.1002/jmri.24619)) formalism provides an elegant and computationally efficient solution for this purpose.

First we set up the necessary code and jupyter environment.

In [34]:
import sys
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as wgt
import tqdm.auto as tqdm
import rich

from IPython.display import display
from pathlib import Path
from numpy.typing import NDArray, ArrayLike
from typing import Literal

In [15]:
%load_ext autoreload
%autoreload 2
%matplotlib widget

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


This should be readily importable when the repository is cloned from GitHub and the notebook is run in its repository defined folder 🚀

In [16]:
src_directory = Path('../src')
init_directory = src_directory / 'initialization'
sys.path.append(str(src_directory))
print = rich.print # nicer outputs

In [17]:
import seqmetrics # noqa: F401
from plotting.splinetools import SplineSettingsDashboard
from plotting.curveditor import MonotonicCurveEditor
from plotting.simulator import SimulationController, SimulationControllerDashboard, SequenceParameters
from plotting.scattercanvas import InteractiveRelaxometricParameterCanvas
from plotting.signaldisplay import SignalDisplayDashboard, SignalDisplay, wire_callbacks
from plotting.historyplot import HistoryPlot
from slsqp import optimize_sequence

Here we load the ncessary flip angle train data and the repetition time pattern from preset numpy arrays.
Both patterns were deduced from the brain-specific work by [Cao et al. 2022](https://doi.org/10.1002/mrm.29194).
They act as an starting point for the interactive modification tooling and subsequently the optimization.
In principle, any other patterns can be loaded and tested here.
Other examples include the:
- original sinusoidal pattern by [Yun et al. 2015](https://doi.org/10.1002/mrm.25559)
- optimized patterns by [Zhao et al. 2019](10.1109/TMI.2018.2873704)

In [18]:
FA_DATA_PATH = init_directory / 'fa_cao.npy'
TR_DATA_PATH = init_directory / 'tr_cao.npy'
fa = np.load(FA_DATA_PATH)
tr = np.load(TR_DATA_PATH)
fa_initial_y = fa
fa_initial_x = np.arange(len(fa))
tr_initial_y = tr
tr_initial_x = np.arange(len(tr))

The interactive spline editors provide an intuitive way to manipulate MR fingerprinting sequences through direct manual adjustment of acquisition parameters.

Flip Angle (FA) Editor:
- Purpose: Design flip angle trajectories across the sequence timepoints
- Controls: Click and drag control points to shape the FA curve
- Range: Typically 0-90 degrees for optimal signal variation
- Tips:
  - Periodic patterns can enhance tissue discrimination
  - Sharp transitions create distinct signal signatures
  - Use reset inside tabs to revert to initial state

Repetition Time (TR) Editor:
- Purpose: Define variable TR patterns for enhanced parameter encoding
- Controls: Click and drag control points to shape the TR curve
- Range: Usually 10-50 ms for rapid fingerprinting sequences
- Benefits:
  - Variable TR improves T1/T2 sensitivity separation
  - Shorter TRs enable faster acquisition

Scatter Canvas Editor
- Purpose: Define relaxometric species
- Controls
  - Leftclick into (T1, T2) grid to add new relaxometric species
  - Rightclick deletes closest species in (T1,T2) grid
  - Species with defined (T1,T2) combinations are automatically utilized in subsequent simulations

In [19]:
dashboard_fa = SplineSettingsDashboard.from_FA_defaults()
dashboard_tr = SplineSettingsDashboard.from_TR_defaults()

seqparams = SequenceParameters(
    ph=np.full_like(fa, fill_value=0.0),
    shots=len(fa),
    prep=[1],
    t2te=[0.0],
    ti=[10.0],
    te=1.0
)

with plt.ioff():
    fig, axes = plt.subplots(ncols=3, figsize=(13.3, 3.9))
    ce_fa = MonotonicCurveEditor(fa_initial_x, fa_initial_y, dashboard_fa, fig=fig, ax=axes[0])
    ce_fa.ax.set_ylabel('Flip Angle (degrees)')
    ce_fa.ax.set_title('Interactive Flip Angle Editor')

    ce_tr = MonotonicCurveEditor(tr_initial_x, tr_initial_y, dashboard_tr, fig=fig, ax=axes[1], initial_yaxis_range=(0, 100))
    ce_tr.ax.set_ylabel('Repetition Time (ms)')
    ce_tr.ax.set_title('Interactive Repetition Time Editor')

    cv = InteractiveRelaxometricParameterCanvas.prepopulated(species={'csf', 'wm', 'gm', 'muscle'}, ax=axes[2], fig=fig)

    ctr_dashboard = SimulationControllerDashboard()

    controller = SimulationController(
        dashboard=ctr_dashboard,
        fa_provider=ce_fa,
        tr_provider=ce_tr,
        relax_provider=cv,
        parameters=seqparams
    )

    tabs = wgt.Tab(
        children=[ctr_dashboard.ui, dashboard_fa.ui, dashboard_tr.ui],
        titles=['Simulation Dashboard', 'FA Settings', 'TR Settings'],
        style={'description_width': 'initial'}
    )

    _ = fig.tight_layout()
    
controller.run()

In [20]:
dashboard = SignalDisplayDashboard.create()
sigdisp = SignalDisplay()
sigdisp.fig.update_layout(height=400, width=1400, showlegend=False)
sigdisp.fig.update_layout(xaxis=dict(title='Time (ms)'), yaxis=dict(title='Signal Amplitude (a.u.)'))

sigdisp.add_traces(controller.fetch_simulation_package(), dashboard.query_state())
wire_callbacks(sigdisp, dashboard, controller)

In [21]:
display(
    wgt.VBox([
        wgt.VBox([tabs, fig.canvas]),
        dashboard.ui,
        sigdisp.fig
    ])
)

With the setup about the relaxometric species from the scatter canvas and the sequence specific parameters {flip angle train, repetition times} from the interactive spline curve obtained above, we can optimize the sequence with respect to different cost functions.

The parameter data defined by the points inside the relaxometric species canvas is reused for the optimization.
Note that the optimization does depend on the currently set species, since the output signal fitness is quantified via the orthogonality cost function.
In turn, this means that we can optimize for specific anatomic regions with prior knowledge about expected tissue
environments and subsequent relaxometric parameters.

In [22]:
T1: NDArray = np.asarray([p.x for p in cv.get_points()], dtype=np.float32)
T2: NDArray = np.asarray([p.y for p in cv.get_points()], dtype=np.float32)
M0: float = 1.0

We also utilize the sequence data from the interactive editors above as a starting point the optimization.

In [ ]:
fa: NDArray = ce_fa.get_current_curve().y
tr: NDArray = ce_tr.get_current_curve().y
ph: float = seqparams.ph

SLSQP support simple box constraints on variables. We utilize this for the flip angle like so: $5^{\circ} \leq \alpha \leq 90^{\circ}$ and $\Delta_{\mathrm{alpha}}=5^{\circ}$. Also the maximum number of iterations determines the overall optimization wall clock duration.

In [49]:
n_max_iter: int = 8
fa_min: float = 5.0
fa_max: float = 90.0
fa_maxdiff: float = 5.0
cost_function: Literal['crlb_sc_iso', 'crlb_sc_epg', 'crlb_mc_iso', 'crlb_mc_epg', 'orth_iso', 'orth_epg'] = 'orth_epg'

In [50]:
# Other sequence parameters (already defined above, but extracted here for clarity)
beats: int = seqparams.beats
shots: int = seqparams.shots
prep: list[int] = seqparams.prep
ti: list[float] = seqparams.ti
t2te: list[float] = seqparams.t2te
te: float = seqparams.te
ph: NDArray = seqparams.ph

In [51]:
init_fa = ce_fa.get_current_curve().y

hp = HistoryPlot(max_TTL=40)
hp.add_immortal_trace(init_fa, width=1, color='green', dash='dot', opacity=0.7)
hp.fig.update_layout(xaxis=dict(title='TR index'), yaxis=dict(title='Flip Angle (degrees)'))

FigureWidget({
    'data': [{'line': {'color': 'green', 'dash': 'dot', 'width': 1},
              'meta': {'TTL': -999, 'is_immortal': True},
              'mode': 'lines',
              'name': 'immortal',
              'opacity': 0.7,
              'type': 'scatter',
              'uid': '39a8be05-8617-4fa5-8bae-5f3eef225b61',
              'y': {'bdata': ('MDHd+rOiJEDRWXRTEEMlQMibzlhi4y' ... '/7QyRAm3HHIuxUJEB7S2vi+2UkQA=='),
                    'dtype': 'f8'}}],
    'layout': {'height': 400,
               'showlegend': True,
               'template': '...',
               'width': 1200,
               'xaxis': {'title': {'text': 'TR index'}},
               'yaxis': {'title': {'text': 'Flip Angle (degrees)'}}}
})

In [52]:
pbar = tqdm.tqdm(total=n_max_iter)

def callback(x: np.ndarray):
    # tgriesler optimizatiopn: single array for fa and tr
    # only propagate flip angles into visualization
    hp.add_trace(x[:x.size//2])
    pbar.update(1)

result = optimize_sequence(
    costfunction=cost_function,
    t1=T1,
    t2=T2,
    m0=M0,
    beats=beats,
    shots=shots,
    fa=init_fa,
    tr=tr,
    ph=ph,
    prep=prep,
    ti=ti,
    t2te=t2te,
    te=te,
    fa_min=fa_min,
    fa_max=fa_max,
    fa_maxdiff=fa_maxdiff,
    n_iter_max=n_max_iter,
    callback=callback,
    iprint=0
)

  0%|          | 0/8 [00:00<?, ?it/s]

In [ ]:
DATA = [t.y for t in hp.fig.data]


In [ ]:
import attrs


def to_list(arraylike: ArrayLike) -> list[float]:
    pass


@attrs.define
class OptimizationPackage:
    """
    Compound data structure to hold all relevant information about an
    optimization run.

    Attributes should respect JSON-serializability for easy saving/loading.
    """
    costfunction: str
    T1: list[float]
    T2: list[float]
    M0: float
    beats: int
    shots: int
    tr: list[float]
    fa_initial: list[float]
    fa_history: list[list[float]]
    ph: list[float]
    prep: list[int]
    ti: list[float]
    t2te: list[float]
    te: float
    fa_min: float
    fa_max: float
    fa_maxdiff: float
    n_iter_max: int

    @classmethod
    def from_preassembled(
        cls,
        costfunction: str,
        T1: NDArray,
        T2: NDArray,
        M0: float,
        sequence_params: SequenceParameters,
        fa_initial: NDArray,
        fa_history: list[NDArray],
        ) -> "OptimizationPackage":
        """
        Convenience constructor that deduces attributes from preassembled sequence
        parameters.
        """
